# GDML

This example reads geometries from [GDML](https://gdml.web.cern.ch) files, the geometry exchange
format of Geant4 and ROOT: test geometries of the shapes and the CMS detector (2018). It draws
them and makes X-ray images of the muon barrel of CMS.

## Loading the modules

In [ ]:
using Geom4hep
using CairoMakie: CairoMakie, Figure, LScene, Axis, heatmap!, DataAspect

The GDML files are in the `examples` directory of the package

In [ ]:
gdmldir = joinpath(pkgdir(Geom4hep), "examples");

## Reading a GDML file
`read_gdml` returns the world volume of the file, with the floating point type given as second
argument (`Float64` by default). Lengths are converted to mm and angles to radians.

In [ ]:
world = read_gdml(joinpath(gdmldir, "polyhedra.gdml"))
[(d.volume.label, d.volume.shape) for d in world.daughters]

The volumes can be drawn as any other hierarchy of volumes:

In [ ]:
fig = Figure(size=(1000, 400))
draw!(LScene(fig[1, 1], show_axis=false), world)
draw!(LScene(fig[1, 2], show_axis=false), read_gdml(joinpath(gdmldir, "torus.gdml")))
draw!(LScene(fig[1, 3], show_axis=false), read_gdml(joinpath(gdmldir, "polycones.gdml")))
fig

## The CMS detector
The GDML file of CMS (2018) has about 20 000 placements of 3 800 volumes. It is read in a few
seconds. Here in single precision, as needed on GPUs.

In [ ]:
cms = read_gdml(joinpath(gdmldir, "cms2018.gdml"), Float32)
[d.volume.label for d in cms[1].daughters][1:8]

The muon system is the 7th daughter of the CMS envelope, and the muon barrel the first daughter of
the muon system. `world_volume` puts it into a world box of its own.

In [ ]:
barrel = cms[1, 7, 1]
world = world_volume(barrel)
fig = Figure(size=(800, 600))
draw!(LScene(fig[1, 1], show_axis=false), world; maxlevel=3)
fig

## X-ray images
The X-ray images integrate the density along rays parallel to the axes, through the navigator
of the closed geometry model.

In [ ]:
model = close_geometry(world)

In [ ]:
fig = Figure(size=(1000, 500))
for axis in (3, 1)
    u, v, image, nsteps = xray(model, axis, 300)
    heatmap!(Axis(fig[1, axis == 3 ? 1 : 2], title="rays along $("xyz"[axis])", aspect=DataAspect()),
             u, v, image, colormap=:grayC)
end
fig

The whole detector (`world_volume(cms)` instead of the barrel) can be processed in the same way.
It is much faster on a GPU: see `examples/XRay.jl` in the repository.

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*